In [1]:
"""
Deepseek Math 7B Training Script - Kaggle H100 Optimized
Includes quick test mode and full training mode with proper model saving
With time tracking, logging, and 12-hour limit enforcement
"""

import os
import sys
import argparse
import inspect
import logging
import time
from datetime import datetime, timedelta
import pandas as pd
import numpy as np
import torch
import joblib
from pathlib import Path
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from transformers import TrainingArguments, Trainer, DataCollatorWithPadding, TrainerCallback
from datasets import Dataset

# ============================================================================
# LOGGING SETUP
# ============================================================================

def setup_logging(log_dir=None):
    """Setup comprehensive logging with timestamps"""
    # Determine log directory
    if log_dir is None:
        if os.path.exists("/kaggle"):
            log_dir = "/kaggle/working/logs"
        else:
            log_dir = "./logs"

    os.makedirs(log_dir, exist_ok=True)
    log_file = os.path.join(log_dir, f"training_{datetime.now().strftime('%Y%m%d_%H%M%S')}.log")

    logging.basicConfig(
        level=logging.INFO,
        format='%(asctime)s | %(levelname)s | %(message)s',
        datefmt='%Y-%m-%d %H:%M:%S',
        handlers=[
            logging.FileHandler(log_file),
            logging.StreamHandler(sys.stdout)
        ]
    )
    logger = logging.getLogger(__name__)
    logger.info(f"Logging to: {log_file}")
    return logger

# ============================================================================
# CONFIGURATION
# ============================================================================

KAGGLE_COMPETITION_PATH = "/kaggle/input/competitions/ai-mathematical-olympiad-progress-prize-3"
KAGGLE_DATASET_PATH = "/kaggle/input/datasets/mkhanshovon43/map-ds/DS/train.csv"
KAGGLE_OUTPUT_PATH = "/kaggle/working/"
KAGGLE_STATE_PATH = "/kaggle/temp"

TEMP_DIR = os.path.join(KAGGLE_STATE_PATH, "deepseek_tmp")
OUTPUT_MODEL_PATH = os.path.join(KAGGLE_OUTPUT_PATH, "saved_model")

# Time limit configuration (12 hours with 30 min buffer for saving)
MAX_TRAINING_HOURS = 11.5  # Leave 30 min buffer for model saving
MAX_TRAINING_SECONDS = MAX_TRAINING_HOURS * 3600

# ============================================================================
# TIME TRACKING CALLBACK
# ============================================================================

class TimeTrackingCallback(TrainerCallback):
    """Callback to track training time and enforce time limits"""

    def __init__(self, max_seconds, logger, save_callback=None):
        self.max_seconds = max_seconds
        self.logger = logger
        self.start_time = None
        self.save_callback = save_callback
        self.last_log_time = 0
        self.log_interval = 300  # Log every 5 minutes

    def on_train_begin(self, args, state, control, **kwargs):
        self.start_time = time.time()
        self.logger.info("=" * 60)
        self.logger.info("TRAINING STARTED")
        self.logger.info(f"Max training time: {self.max_seconds/3600:.1f} hours")
        self.logger.info(f"Start time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
        self.logger.info("=" * 60)

    def on_step_end(self, args, state, control, **kwargs):
        elapsed = time.time() - self.start_time
        remaining = self.max_seconds - elapsed

        # Log progress every 5 minutes
        if elapsed - self.last_log_time >= self.log_interval:
            self.last_log_time = elapsed
            elapsed_str = str(timedelta(seconds=int(elapsed)))
            remaining_str = str(timedelta(seconds=int(remaining)))
            progress_pct = (state.global_step / state.max_steps * 100) if state.max_steps else 0

            self.logger.info(
                f"Step {state.global_step}/{state.max_steps} ({progress_pct:.1f}%) | "
                f"Elapsed: {elapsed_str} | Remaining: {remaining_str}"
            )

        # Check time limit - stop training if approaching limit
        if remaining < 1800:  # 30 minutes remaining
            self.logger.warning("=" * 60)
            self.logger.warning("APPROACHING TIME LIMIT - STOPPING TRAINING EARLY")
            self.logger.warning(f"Elapsed: {elapsed/3600:.2f} hours")
            self.logger.warning("Saving model before timeout...")
            self.logger.warning("=" * 60)
            control.should_training_stop = True
            control.should_save = True

        return control

    def on_epoch_end(self, args, state, control, **kwargs):
        elapsed = time.time() - self.start_time
        remaining = self.max_seconds - elapsed
        elapsed_str = str(timedelta(seconds=int(elapsed)))
        remaining_str = str(timedelta(seconds=int(remaining)))

        self.logger.info("=" * 60)
        self.logger.info(f"EPOCH {state.epoch:.0f} COMPLETED")
        self.logger.info(f"Elapsed time: {elapsed_str}")
        self.logger.info(f"Remaining time: {remaining_str}")
        self.logger.info(f"Training loss: {state.log_history[-1].get('loss', 'N/A') if state.log_history else 'N/A'}")
        self.logger.info("=" * 60)

    def on_train_end(self, args, state, control, **kwargs):
        elapsed = time.time() - self.start_time
        elapsed_str = str(timedelta(seconds=int(elapsed)))

        self.logger.info("=" * 60)
        self.logger.info("TRAINING COMPLETED")
        self.logger.info(f"Total training time: {elapsed_str}")
        self.logger.info(f"Total steps: {state.global_step}")
        self.logger.info(f"End time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
        self.logger.info("=" * 60)


def setup_environment(quick_run=False, logger=None):
    """Setup directories and environment variables"""
    os.makedirs(KAGGLE_STATE_PATH, exist_ok=True)
    os.makedirs(TEMP_DIR, exist_ok=True)
    os.makedirs(OUTPUT_MODEL_PATH, exist_ok=True)
    os.makedirs(f"{TEMP_DIR}/logs/", exist_ok=True)

    os.environ["HF_HOME"] = os.path.join(KAGGLE_STATE_PATH, "hf_home")
    os.environ["TRANSFORMERS_CACHE"] = os.path.join(KAGGLE_STATE_PATH, "hf_home", "transformers")
    os.environ["HF_DATASETS_CACHE"] = os.path.join(KAGGLE_STATE_PATH, "hf_home", "datasets")
    os.environ["CUDA_VISIBLE_DEVICES"] = "0"

    log = logger or logging.getLogger(__name__)

    # Check available disk space
    try:
        import shutil
        for path, name in [(KAGGLE_OUTPUT_PATH, "Output"), (KAGGLE_STATE_PATH, "Temp")]:
            if os.path.exists(path):
                total, used, free = shutil.disk_usage(path)
                log.info(f"Disk space ({name}): {free/1e9:.1f} GB free / {total/1e9:.1f} GB total")
    except Exception as e:
        log.warning(f"Could not check disk space: {e}")

    log.info(f"GPU Available: {torch.cuda.is_available()}")
    if torch.cuda.is_available():
        log.info(f"GPU Name: {torch.cuda.get_device_name(0)}")
        log.info(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

# ============================================================================
# DATA LOADING AND PREPROCESSING
# ============================================================================

def load_and_prepare_data(quick_run=False, logger=None):
    """Load data from Kaggle input or local path"""
    log = logger or logging.getLogger(__name__)
    log.info("Loading training data...")

    candidate_paths = [
        KAGGLE_DATASET_PATH,
        os.path.join(KAGGLE_COMPETITION_PATH, "train.csv"),
        "train.csv",
    ]

    csv_path = next((p for p in candidate_paths if os.path.exists(p)), None)
    if csv_path is None:
        raise FileNotFoundError(
            "train.csv not found. Checked dataset and competition paths."
        )

    log.info(f"Loading from: {csv_path}")
    train = pd.read_csv(csv_path)

    if quick_run:
        train = train.sample(n=min(1000, len(train)), random_state=42)
        log.info(f"Quick run mode: Using {len(train)} samples for testing")

    train.Misconception = train.Misconception.fillna('NA')
    train['target'] = train.Category + ":" + train.Misconception

    le = LabelEncoder()
    train['label'] = le.fit_transform(train['target'])
    n_classes = len(le.classes_)

    log.info(f"Train shape: {train.shape} with {n_classes} target classes")
    log.info(f"Sample data:\n{train.head()}")

    return train, le, n_classes

def process_labels(train, logger=None):
    """Process and mark correct answers"""
    log = logger or logging.getLogger(__name__)
    log.info("Processing labels and marking correct answers...")

    # FIX: use str.split safely; Category may not always contain '_'
    idx = train['Category'].apply(lambda c: c.split('_')[0]) == 'True'
    correct = train.loc[idx].copy()
    correct['c'] = correct.groupby(['QuestionId', 'MC_Answer'])['MC_Answer'].transform('count')
    correct = correct.sort_values('c', ascending=False)
    correct = correct.drop_duplicates(['QuestionId'])
    correct = correct[['QuestionId', 'MC_Answer']]
    correct['is_correct'] = 1

    train = train.merge(correct, on=['QuestionId', 'MC_Answer'], how='left')
    train['is_correct'] = train['is_correct'].fillna(0)

    log.info(f"Label processing complete. Correct answers marked: {train['is_correct'].sum():.0f}")
    return train

def format_input(row):
    """Format input text for model"""
    x = "Yes" if row['is_correct'] else "No"
    return (
        f"Question: {row['QuestionText']}\n"
        f"Answer: {row['MC_Answer']}\n"
        f"Correct? {x}\n"
        f"Student Explanation: {row['StudentExplanation']}"
    )

# ============================================================================
# MODEL AND TOKENIZER SETUP
# ============================================================================

def load_model_and_tokenizer(n_classes, logger=None):
    """Load model and tokenizer with optimized settings"""
    log = logger or logging.getLogger(__name__)
    log.info("Loading model and tokenizer...")
    start_time = time.time()

    Model_Name = "deepseek-ai/deepseek-math-7b-instruct"

    try:
        model = AutoModelForSequenceClassification.from_pretrained(
            Model_Name,
            num_labels=n_classes,
            torch_dtype=torch.bfloat16,
            device_map="auto",
            cache_dir=TEMP_DIR,
            low_cpu_mem_usage=True,
        )
    except Exception as e:
        log.error(f"Failed to load model: {e}")
        raise

    tokenizer = AutoTokenizer.from_pretrained(Model_Name, cache_dir=TEMP_DIR)

    if tokenizer.pad_token is None:
        tokenizer.add_special_tokens({'pad_token': '[PAD]'})
        model.resize_token_embeddings(len(tokenizer))
        model.config.pad_token_id = tokenizer.pad_token_id

    load_time = time.time() - start_time
    log.info(f"Model loaded successfully in {load_time:.1f} seconds!")
    return model, tokenizer

def verify_model_loaded(model, tokenizer, logger=None):
    """Run a tiny forward pass to prove model weights are loaded and usable."""
    log = logger or logging.getLogger(__name__)
    log.info("Verifying model load with a tiny forward pass...")
    probe_text = "Question: 1+1?\nAnswer: 2\nCorrect? Yes\nStudent Explanation: basic arithmetic"
    encoded = tokenizer(probe_text, return_tensors="pt", truncation=True, max_length=64)
    model_device = next(model.parameters()).device
    encoded = {k: v.to(model_device) for k, v in encoded.items()}

    with torch.no_grad():
        out = model(**encoded)

    log.info(
        "Model load verified: "
        f"logits_shape={tuple(out.logits.shape)}, "
        f"device={model_device}, "
        f"dtype={next(model.parameters()).dtype}"
    )

# ============================================================================
# DATASET PREPARATION
# ============================================================================

def prepare_dataset(train, tokenizer, logger=None):
    """Prepare train/validation HuggingFace datasets with tokenization"""
    log = logger or logging.getLogger(__name__)
    log.info("Preparing dataset...")
    start_time = time.time()

    train['text'] = train.apply(format_input, axis=1)
    log.info("Example prompt:")
    log.info(train['text'].values[0])

    COLS = ['text', 'label']
    full_df = train[COLS].copy()
    full_df['label'] = full_df['label'].astype(np.int64)
    full_df = full_df.reset_index(drop=True)

    stratify_labels = full_df['label']
    if full_df['label'].value_counts().min() < 2:
        stratify_labels = None
        log.warning("Some classes have <2 samples; using non-stratified validation split.")

    train_df, eval_df = train_test_split(
        full_df,
        test_size=0.1,
        random_state=42,
        stratify=stratify_labels,
    )

    train_ds = Dataset.from_pandas(train_df.reset_index(drop=True), preserve_index=False)
    eval_ds = Dataset.from_pandas(eval_df.reset_index(drop=True), preserve_index=False)

    def tokenize(batch):
        return tokenizer(batch["text"], truncation=True, max_length=256)

    num_proc = min(4, os.cpu_count() or 1)
    train_ds = train_ds.map(tokenize, batched=True, remove_columns=['text'], num_proc=num_proc)
    eval_ds = eval_ds.map(tokenize, batched=True, remove_columns=['text'], num_proc=num_proc)

    prep_time = time.time() - start_time
    log.info(f"Dataset prepared in {prep_time:.1f} seconds")
    log.info(f"Train samples: {len(train_ds)} | Validation samples: {len(eval_ds)}")
    return train_ds, eval_ds


def compute_metrics(eval_pred):
    """Compute validation metrics shown in logs."""
    logits, labels = eval_pred

    preds = np.argmax(logits, axis=-1)
    accuracy = (preds == labels).mean().item()

    shifted = logits - np.max(logits, axis=1, keepdims=True)
    exp_scores = np.exp(shifted)
    probs = exp_scores / np.sum(exp_scores, axis=1, keepdims=True)
    top3 = np.argsort(-probs, axis=1)[:, :3]

    map3_sum = 0.0
    for i, label in enumerate(labels):
        if top3[i, 0] == label:
            map3_sum += 1.0
        elif top3[i, 1] == label:
            map3_sum += 0.5
        elif top3[i, 2] == label:
            map3_sum += 1.0 / 3.0

    map3 = map3_sum / len(labels)
    return {"accuracy": accuracy, "map@3": map3}

# ============================================================================
# TRAINING
# ============================================================================

def get_training_args(quick_run=False, logger=None):
    """Get optimized training arguments - Kaggle disk-optimized"""
    log = logger or logging.getLogger(__name__)
    ta_params = inspect.signature(TrainingArguments.__init__).parameters
    eval_key = "evaluation_strategy" if "evaluation_strategy" in ta_params else "eval_strategy"

    if quick_run:
        log.info("Using QUICK RUN training arguments (limited epochs/steps)")
        quick_kwargs = {
            "output_dir": OUTPUT_MODEL_PATH,  # Save directly to output
            "do_train": True,
            "do_eval": False,  # Disable eval to save memory
            "save_strategy": "no",  # NO checkpoints - saves disk space
            "num_train_epochs": 1,
            "per_device_train_batch_size": 16,
            "learning_rate": 5e-5,
            "logging_dir": f"{TEMP_DIR}/logs/",
            "logging_steps": 50,
            "bf16": True,
            "fp16": False,
            "report_to": "none",
            "warmup_ratio": 0.05,
            "lr_scheduler_type": "linear",
            "dataloader_pin_memory": False,  # Reduce memory pressure
            "gradient_checkpointing": True,
            "max_steps": 100,
        }
        quick_kwargs[eval_key] = "no"
        return TrainingArguments(**quick_kwargs)
    else:
        log.info("Using FULL TRAINING arguments (3 epochs) - Kaggle disk optimized")
        log.info("NOTE: No intermediate checkpoints to save disk space")
        full_kwargs = {
            "output_dir": OUTPUT_MODEL_PATH,  # Save directly to /kaggle/working
            "do_train": True,
            "do_eval": False,  # Disable eval - saves memory & disk
            "save_strategy": "no",  # NO intermediate checkpoints - critical for Kaggle disk limits
            "num_train_epochs": 3,
            "per_device_train_batch_size": 16,  # Reduced from 32 for memory
            "learning_rate": 5e-5,
            "logging_dir": f"{TEMP_DIR}/logs/",
            "logging_steps": 500,
            "bf16": True,
            "fp16": False,
            "report_to": "none",
            "warmup_ratio": 0.1,
            "lr_scheduler_type": "cosine",
            "dataloader_pin_memory": False,  # Reduce memory pressure
            "gradient_checkpointing": True,
            "optim": "adamw_torch",
            "weight_decay": 0.01,
        }
        full_kwargs[eval_key] = "no"
        return TrainingArguments(**full_kwargs)

def train_model(model, tokenizer, train_ds, eval_ds, quick_run=False, logger=None):
    """Train the model with time tracking"""
    log = logger or logging.getLogger(__name__)
    log.info("Initializing trainer...")

    training_args = get_training_args(quick_run=quick_run, logger=log)
    data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

    # Create time tracking callback
    time_callback = TimeTrackingCallback(
        max_seconds=MAX_TRAINING_SECONDS,
        logger=log
    )

    # Handle compatibility between old 'tokenizer' and new 'processing_class' parameter
    ta_params = inspect.signature(Trainer.__init__).parameters
    trainer_kwargs = {
        "model": model,
        "args": training_args,
        "train_dataset": train_ds,
        # No eval_dataset - disabled to save disk space on Kaggle
        "data_collator": data_collator,
        "callbacks": [time_callback],
    }

    # Use 'processing_class' if available (newer transformers), else 'tokenizer'
    if "processing_class" in ta_params:
        trainer_kwargs["processing_class"] = tokenizer
    else:
        trainer_kwargs["tokenizer"] = tokenizer

    trainer = Trainer(**trainer_kwargs)

    log.info("Starting training loop...")
    try:
        trainer.train()
    except Exception as e:
        log.error(f"Training error: {e}")
        log.info("Attempting to save model despite error...")
        raise

    log.info("=" * 60)
    log.info("TRAINING LOOP COMPLETED SUCCESSFULLY")
    log.info("=" * 60)

    return trainer

# ============================================================================
# MODEL SAVING
# ============================================================================

def clear_disk_space(logger=None):
    """Clear cache and temp files to free disk space before saving"""
    log = logger or logging.getLogger(__name__)
    log.info("Clearing disk space before saving model...")

    import gc
    import shutil

    # Clear Python garbage
    gc.collect()

    # Clear CUDA cache
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        log.info("Cleared CUDA cache")

    # Clear HuggingFace cache (model downloads - we don't need them after loading)
    cache_dirs = [
        os.path.join(TEMP_DIR, "models--deepseek-ai--deepseek-math-7b-instruct"),
        os.environ.get("TRANSFORMERS_CACHE", ""),
        os.environ.get("HF_DATASETS_CACHE", ""),
    ]

    for cache_dir in cache_dirs:
        if cache_dir and os.path.exists(cache_dir):
            try:
                # Get size before deletion
                size = sum(os.path.getsize(os.path.join(dirpath, filename))
                          for dirpath, _, filenames in os.walk(cache_dir)
                          for filename in filenames)
                shutil.rmtree(cache_dir, ignore_errors=True)
                log.info(f"Cleared cache: {cache_dir} ({size/1e9:.2f} GB freed)")
            except Exception as e:
                log.warning(f"Could not clear {cache_dir}: {e}")

    gc.collect()


def save_model_and_artifacts(trainer, tokenizer, le, quick_run=False, save_quick_model=True, logger=None):
    """Save trained model and label encoder - ALWAYS saves full model"""
    log = logger or logging.getLogger(__name__)
    save_dir = OUTPUT_MODEL_PATH
    log.info("=" * 60)
    log.info("SAVING MODEL AND ARTIFACTS")
    log.info(f"Output directory: {save_dir}")
    log.info("=" * 60)

    # Clear disk space first - critical for Kaggle
    clear_disk_space(logger=log)

    start_time = time.time()
    os.makedirs(save_dir, exist_ok=True)

    # ALWAYS save the full model (regardless of quick_run mode)
    try:
        model_save_path = f"{save_dir}/deepseek_model"
        log.info(f"Saving model to: {model_save_path}")
        trainer.save_model(model_save_path)
        log.info("Model saved successfully!")
    except Exception as e:
        log.error(f"Failed to save model via trainer: {e}")
        # Fallback: try saving directly
        try:
            log.info("Attempting fallback save method...")
            trainer.model.save_pretrained(model_save_path)
            log.info("Model saved via fallback method!")
        except Exception as e2:
            log.error(f"Fallback save also failed: {e2}")
            raise

    # Save label encoder
    try:
        le_save_path = f"{save_dir}/label_encoder.joblib"
        joblib.dump(le, le_save_path)
        log.info(f"Label encoder saved to: {le_save_path}")
    except Exception as e:
        log.error(f"Failed to save label encoder: {e}")

    # Save tokenizer
    try:
        tokenizer_save_path = f"{save_dir}/tokenizer"
        os.makedirs(tokenizer_save_path, exist_ok=True)
        tokenizer.save_pretrained(tokenizer_save_path)
        trainer.model.config.to_json_file(f"{tokenizer_save_path}/config.json")
        log.info(f"Tokenizer/config saved to: {tokenizer_save_path}")
    except Exception as e:
        log.error(f"Failed to save tokenizer: {e}")

    save_time = time.time() - start_time
    log.info("=" * 60)
    log.info(f"All artifacts saved in {save_time:.1f} seconds!")
    log.info(f"Output directory: {save_dir}")
    log.info("=" * 60)

    # List saved files
    try:
        for root, dirs, files in os.walk(save_dir):
            for f in files:
                filepath = os.path.join(root, f)
                size_mb = os.path.getsize(filepath) / (1024 * 1024)
                log.info(f"  Saved: {filepath} ({size_mb:.1f} MB)")
    except Exception:
        pass

# ============================================================================
# MAIN EXECUTION
# ============================================================================

def main():
    """Main execution function"""
    # Setup logging first (auto-detects Kaggle vs local)
    logger = setup_logging()

    overall_start_time = time.time()
    logger.info("=" * 80)
    logger.info("DEEPSEEK MATH 7B TRAINING SCRIPT")
    logger.info(f"Script started at: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    logger.info(f"Max allowed training time: {MAX_TRAINING_HOURS} hours")
    logger.info("=" * 80)

    parser = argparse.ArgumentParser(description="Deepseek Math 7B Training for Kaggle")
    parser.add_argument("--quick", action="store_true", help="Run quick test mode (1 epoch, limited data)")
    parser.add_argument("--full", action="store_true", help="Run full training mode (3 epochs)")
    parser.add_argument(
        "--save-quick-model",
        action="store_true",
        help="In quick mode, also save full model weights (uses significant disk)",
    )

    args, unknown_args = parser.parse_known_args()
    if unknown_args:
        logger.info(f"Ignoring notebook/kernel args: {unknown_args}")

    # Full training is the default; --quick activates smoke-test mode.
    quick_run = args.quick and not args.full
    save_quick_model = True  # Always save the full model

    if quick_run:
        logger.info("=" * 80)
        logger.info("QUICK RUN MODE - Fast smoke test with validation metrics")
        logger.info("=" * 80)
    else:
        logger.info("=" * 80)
        logger.info("FULL TRAINING MODE - Training with validation monitoring")
        logger.info("=" * 80)

    try:
        setup_environment(quick_run=quick_run, logger=logger)

        train, le, n_classes = load_and_prepare_data(quick_run=quick_run, logger=logger)
        train = process_labels(train, logger=logger)

        model, tokenizer = load_model_and_tokenizer(n_classes, logger=logger)

        if quick_run:
            verify_model_loaded(model, tokenizer, logger=logger)

        train_ds, eval_ds = prepare_dataset(train, tokenizer, logger=logger)

        trainer = train_model(model, tokenizer, train_ds, eval_ds, quick_run=quick_run, logger=logger)

        save_model_and_artifacts(
            trainer,
            tokenizer,
            le,
            quick_run=quick_run,
            save_quick_model=save_quick_model,
            logger=logger,
        )

        total_time = time.time() - overall_start_time
        total_time_str = str(timedelta(seconds=int(total_time)))

        logger.info("")
        logger.info("=" * 80)
        if quick_run:
            logger.info("QUICK TEST COMPLETED SUCCESSFULLY!")
            logger.info("Validation metrics are printed above.")
            logger.info("Run with --full flag for full training mode")
        else:
            logger.info("FULL TRAINING COMPLETED SUCCESSFULLY!")
            logger.info("Model saved and ready for evaluation")
        logger.info(f"Total execution time: {total_time_str}")
        logger.info(f"Completed at: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
        logger.info("=" * 80)

    except Exception as e:
        logger.error("=" * 80)
        logger.error(f"TRAINING FAILED WITH ERROR: {e}")
        logger.error("=" * 80)
        import traceback
        logger.error(traceback.format_exc())

        # Try to save whatever we have
        total_time = time.time() - overall_start_time
        logger.info(f"Time elapsed before failure: {str(timedelta(seconds=int(total_time)))}")
        raise

if __name__ == "__main__":
    main()

2026-03-20 15:24:39 | INFO | Logging to: /kaggle/working/logs/training_20260320_152439.log
2026-03-20 15:24:39 | INFO | ================================================================================
2026-03-20 15:24:39 | INFO | DEEPSEEK MATH 7B TRAINING SCRIPT
2026-03-20 15:24:39 | INFO | Script started at: 2026-03-20 15:24:39
2026-03-20 15:24:39 | INFO | Max allowed training time: 11.5 hours
2026-03-20 15:24:39 | INFO | ================================================================================
2026-03-20 15:24:39 | INFO | Ignoring notebook/kernel args: ['-f', '/tmp/tmp92o9lfgw.json', '--HistoryManager.hist_file=:memory:']
2026-03-20 15:24:39 | INFO | ================================================================================
2026-03-20 15:24:39 | INFO | FULL TRAINING MODE - Training with validation monitoring
2026-03-20 15:24:39 | INFO | ================================================================================
2026-03-20 15:24:39 | INFO | Disk space (Output): 20.9 

config.json:   0%|          | 0.00/594 [00:00<?, ?B/s]

2026-03-20 15:24:40 | INFO | HTTP Request: HEAD https://huggingface.co/deepseek-ai/deepseek-math-7b-instruct/resolve/main/adapter_config.json "HTTP/1.1 404 Not Found"


2026-03-20 15:24:40 | WARNING | Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.


`torch_dtype` is deprecated! Use `dtype` instead!


2026-03-20 15:24:40 | INFO | HTTP Request: HEAD https://huggingface.co/deepseek-ai/deepseek-math-7b-instruct/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-03-20 15:24:40 | INFO | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/deepseek-ai/deepseek-math-7b-instruct/0a5828f800a36df0fd7f0ed581b983246c0677ff/config.json "HTTP/1.1 200 OK"
2026-03-20 15:24:40 | INFO | HTTP Request: HEAD https://huggingface.co/deepseek-ai/deepseek-math-7b-instruct/resolve/main/adapter_config.json "HTTP/1.1 404 Not Found"
2026-03-20 15:24:40 | INFO | HTTP Request: HEAD https://huggingface.co/deepseek-ai/deepseek-math-7b-instruct/resolve/main/model.safetensors "HTTP/1.1 404 Not Found"
2026-03-20 15:24:40 | INFO | HTTP Request: HEAD https://huggingface.co/deepseek-ai/deepseek-math-7b-instruct/resolve/main/model.safetensors.index.json "HTTP/1.1 404 Not Found"
2026-03-20 15:24:40 | INFO | HTTP Request: HEAD https://huggingface.co/deepseek-ai/deepseek-math-7b-instruct/resolve/ma

... (17 outputs left out)

Step,Training Loss
500,1.671320
1000,0.616526
1500,0.445620
2000,0.384760
2500,0.257170
3000,0.224511
3500,0.218565
4000,0.213409
4500,0.122901
5000,0.108795


2026-03-20 15:29:56 | INFO | Step 413/6195 (6.7%) | Elapsed: 0:05:00 | Remaining: 11:24:59
2026-03-20 15:34:56 | INFO | Step 1233/6195 (19.9%) | Elapsed: 0:10:00 | Remaining: 11:19:59
2026-03-20 15:39:42 | INFO | ============================================================
2026-03-20 15:39:42 | INFO | EPOCH 1 COMPLETED
2026-03-20 15:39:42 | INFO | Elapsed time: 0:14:45
2026-03-20 15:39:42 | INFO | Remaining time: 11:15:14
2026-03-20 15:39:42 | INFO | Training loss: 0.3847597351074219
2026-03-20 15:39:42 | INFO | ============================================================
2026-03-20 15:39:57 | INFO | Step 2105/6195 (34.0%) | Elapsed: 0:15:00 | Remaining: 11:14:59
2026-03-20 15:44:57 | INFO | Step 3030/6195 (48.9%) | Elapsed: 0:20:00 | Remaining: 11:09:59
2026-03-20 15:49:57 | INFO | Step 3954/6195 (63.8%) | Elapsed: 0:25:00 | Remaining: 11:04:59
2026-03-20 15:50:57 | INFO | ============================================================
2026-03-20 15:50:57 | INFO | EPOCH 2 COMPLETED
2026-

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

2026-03-20 16:02:36 | INFO | Model saved successfully!
2026-03-20 16:02:36 | INFO | Label encoder saved to: /kaggle/working/saved_model/label_encoder.joblib
2026-03-20 16:02:36 | INFO | Tokenizer/config saved to: /kaggle/working/saved_model/tokenizer
2026-03-20 16:02:36 | INFO | ============================================================
2026-03-20 16:02:36 | INFO | All artifacts saved in 22.9 seconds!
2026-03-20 16:02:36 | INFO | Output directory: /kaggle/working/saved_model
2026-03-20 16:02:36 | INFO | ============================================================
2026-03-20 16:02:36 | INFO |   Saved: /kaggle/working/saved_model/label_encoder.joblib (0.0 MB)
2026-03-20 16:02:36 | INFO |   Saved: /kaggle/working/saved_model/deepseek_model/tokenizer_config.json (0.0 MB)
2026-03-20 16:02:36 | INFO |   Saved: /kaggle/working/saved_model/deepseek_model/tokenizer.json (7.2 MB)
2026-03-20 16:02:36 | INFO |   Saved: /kaggle/working/saved_model/deepseek_model/training_args.bin (0.0 MB)
2026-03